In [2]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms, models 
import matplotlib.pyplot as plt 
import os 

In [3]:
# Pretrained models expect ImageNet normalization 

transform_train = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.RandomHorizontalFlip(), # data augmentation 
    transforms.RandomRotation(10),  # 10 degrees
    transforms.ToTensor(), 
    transforms.Normalize(mean = [0.485, 0.456, 0.406], 
                        std = [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224,224)), 
    transforms.ToTensor(), 
    transforms.Normalize(mean = [0.485, 0.456, 0.406], 
                        std = [0.229, 0.224, 0.225])
])

In [4]:
# Load dataset 
data_dir = "/kaggle/input/datasets/puneet6060/intel-image-classification/seg_train/seg_train"
val_dir = "/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test/seg_test" 

train_dataset = datasets.ImageFolder(data_dir, transform = transform_train) 
val_dataset = datasets.ImageFolder(val_dir, transform = transform_val) 


train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True) 
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = False) 


print(f'Classes {train_dataset.classes}')
print(f'Train size : {len(train_dataset)}')
print(f'Val size : {len(val_dataset)}') 



Classes ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Train size : 14034
Val size : 3000


In [5]:
# load pretrained Resnet-18
model = models.resnet18(weights='IMAGENET1K_V1') 

#freeze all layers
for param in model.parameters():
    param.requires_grad = False 

#replace the final one with one matching our 6 classes 

num_features = model.fc.in_features  # 512 

model.fc = nn.Linear(num_features, 6) 

model = model.to('cuda') 

print("Model Ready Trainable params:", 
     sum(p.numel() for p in model.parameters() if p.requires_grad))


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 206MB/s]


Model Ready Trainable params: 3078


In [6]:
# loass and optimizer 

criterion = nn.CrossEntropyLoss() 
optimizer = optim.Adam(model.fc.parameters(), lr=0.001) # only pass parameters that require gradient

In [7]:
def train_one_epoch(model,loader ,criterion, optimizer):
    model.train()
    total_loss, correct = 0,0
    for images, labels in loader:
        images, labels = images.to('cuda'), labels.to('cuda')
        optimizer.zero_grad() 
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() 
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss/ len(loader), correct / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0,0 
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to('cuda'), labels.to('cuda') 
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset) 

In [8]:
# Phase 1 Feature Extraction 

for epoch in range(5):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_acc = evaluate(model, val_loader, criterion)
    print(f'Epoch {epoch + 1} | Train Acc {tr_acc:.3f}| Val Acc: {va_acc:.2f}') 

Epoch 1 | Train Acc 0.812| Val Acc: 0.89
Epoch 2 | Train Acc 0.877| Val Acc: 0.90
Epoch 3 | Train Acc 0.881| Val Acc: 0.88
Epoch 4 | Train Acc 0.886| Val Acc: 0.90
Epoch 5 | Train Acc 0.887| Val Acc: 0.91


In [9]:
# Phase 2: Fine-tuning (unfreeze all, Lower LR) 

for param in model.parameters():
    param.requires_grad = True 

optimizer = optim.Adam(model.parameters(), lr=0.0001) 

for epoch in range(5):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_acc = evaluate(model, val_loader, criterion)
    print(f'Epoch {epoch + 1} | Train Acc {tr_acc:.3f}| Val Acc: {va_acc:.2f}') 

Epoch 1 | Train Acc 0.897| Val Acc: 0.91
Epoch 2 | Train Acc 0.934| Val Acc: 0.93
Epoch 3 | Train Acc 0.946| Val Acc: 0.93
Epoch 4 | Train Acc 0.958| Val Acc: 0.93
Epoch 5 | Train Acc 0.962| Val Acc: 0.92


In [10]:
# save the model 

torch.save(model.state_dict(), 'resnet18_intel.pth') 
print("Model saved") 

Model saved


Q1) Why do we use different learning rates in Phase 1 vs Phase 2?

- In Phase 1, we only train the final layer, which is the new classifier. Therefore, higher learning rates are permitted. However, in Phase 2, we train the entire network with its pre-trained weights, so we must be careful not to disrupt those pre-trained weights.

Q2 ) Why does ImageFolder expect a specific folder structure?
- Because ImageFolder automatically assign labels based on subfolder names, it expects each class to have its own directory. 

Q3 ) What would happen if we forgot to freeze layers in Phase 1?
- The optimiser will update all the network weights.
- The pretrained features may be modified too much, especially with higher LR.
- Training will be slower and requires more memory
- - on a small dataset (like this), the model will likely overfit. 